# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
%pip -q install duckdb huggingface_hub

In [17]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [18]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

One row represents the daily performance of one content item for one client on one report date. The full available time window in the daily fact_table is from 27-01-2025 to 30-06-2026

In [19]:
# Check the grain and the data rangeof the daily fact table
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT
  client_hash_id) AS clients,
    COUNT(DISTINCT
  content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
""").df()
result


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,clients,contents,min_date,max_date
0,78835655,70,427292,2025-01-27,2026-06-30


## 2. Fields: feature / label / context / excluded

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, and scroll_events.

Label: No label is taken directly from the daily fact table. Any prediction label should be derived from a future outcome so that it is not available at feature time.

Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, and month.

Excluded: gsc_sum_position is excluded because gsc_avg_position is the directly interpretable position feature, while the sum is mainly an intermediate aggregate. Identifier fields are also kept as context rather than model features.

In [20]:
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {TABLES['fact_daily']}
""").df()
schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

The data is at daily client-content level, with report_date identifying the observation date. The dataset contains 78,835,655 rows covering 70 clients and 427,292 content items, from 2025-01-27 to 2026-06-30. Required identifiers and dates are not missing. However, GSC and GA4 availability varies across rows, so these fields should be treated as availability indicators rather than assuming that the data exists for every observation.

In [21]:
missing_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_id,
    COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS missing_gsc_availability,
    COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS missing_ga4_availability
FROM {TABLES['fact_daily']}
""").df()

missing_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_report_date,missing_client_id,missing_content_id,missing_gsc_availability,missing_ga4_availability
0,78835655,0,0,0,98006,29635327


In [22]:
march_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

march_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [23]:
gsc_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_with_gsc,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

gsc_check

,rows_with_gsc,gsc_available
0,9841378,3611061


In [24]:
ga4_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_in_march,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

ga4_check

,rows_in_march,ga4_available
0,9841378,413966


## 4. Data limits

The dataset has some important limitations. Client history is unbalanced, so different clients may have different amounts of historical data. GSC and GA4 data are not available for every observation, so some rows have incomplete measurement coverage. Rolling time windows may also overlap, meaning nearby observations may share part of the same historical period. These limitations should be considered when building features and evaluating a model.

In [25]:
limits_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available = TRUE
        AND ga4_data_available = FALSE
    ) AS gsc_only_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available = FALSE
        AND ga4_data_available = TRUE
    ) AS ga4_only_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available = FALSE
        AND ga4_data_available = FALSE
    ) AS neither_available_rows
FROM {TABLES['fact_daily']}
""").df()

limits_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_only_rows,ga4_only_rows,neither_available_rows
0,78835655,17373783,362385,28912779


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.